In [ ]:
import pandas as pd
import numpy as np

def safe_division(numerator, denominator):
    """
    Safely perform division, returning 0 if denominator is 0
    """
    try:
        return numerator / denominator if denominator != 0 else 0
    except:
        return 0

def calculate_evidence_based_likelihoods(dataset):
    """
    Calculate likelihoods for all variables based on clinical thresholds
    from peer-reviewed literature
    """
    # Total number of metabolic syndrome cases
    total_cases = len(dataset[dataset['MetabolicSyndrome'] == 1])
    total_samples = len(dataset)
    
    # Separate male and female subsets for metabolic syndrome
    male_ms_cases = len(dataset[(dataset['MetabolicSyndrome'] == 1) & (dataset['Sex'] == 0)])
    female_ms_cases = len(dataset[(dataset['MetabolicSyndrome'] == 1) & (dataset['Sex'] == 1)])
    
    likelihoods = {}
    
    # Age Likelihoods
    # Ref: Ford ES, et al. Diabetes Care. 2002;25(10):1790-1795
    # Risk increases significantly after age 40 for men
    likelihoods['age_male_risk'] = safe_division(
        len(dataset[(dataset['MetabolicSyndrome'] == 1) & (dataset['Sex'] == 0) & (dataset['Age'] >= 40)]),
        total_cases
    )
    
    # Ref: Carr MC. J Clin Endocrinol Metab. 2003;88(6):2404-2411
    # Post-menopausal women (average age 51) show increased risk
    likelihoods['age_female_risk'] = safe_division(
        len(dataset[(dataset['MetabolicSyndrome'] == 1) & (dataset['Sex'] == 1) & (dataset['Age'] >= 51)]),
        total_cases
    )
    
    # Race Likelihoods
    # Ref: Park YW, et al. Arch Intern Med. 2003;163(4):427-436
    race_mapping = {0: 'White', 1: 'Asian', 2: 'Black', 3: 'MexAmerican', 4: 'Hispanic', 5: 'Other'}
    
    for race_code, race_name in race_mapping.items():
        likelihoods[f'race_{race_name.lower()}'] = safe_division(
            len(dataset[(dataset['MetabolicSyndrome'] == 1) & (dataset['Race'] == race_code)]),
            total_cases
        )
    
    # BMI Likelihood
    # Ref: WHO Technical Report Series 894. Geneva: WHO, 2000
    # BMI ≥ 30 kg/m² indicates obesity and increased metabolic risk
    likelihoods['bmi_risk'] = safe_division(
        len(dataset[(dataset['MetabolicSyndrome'] == 1) & (dataset['BMI'] >= 30)]),
        total_cases
    )
    
    # Waist Circumference Likelihoods
    # Ref: IDF Consensus Worldwide Definition of the Metabolic Syndrome, 2006
    likelihoods['waist_male_risk'] = safe_division(
        len(dataset[(dataset['MetabolicSyndrome'] == 1) & (dataset['Sex'] == 0) & (dataset['WaistCirc'] >= 94)]),
        male_ms_cases
    )
    
    likelihoods['waist_female_risk'] = safe_division(
        len(dataset[(dataset['MetabolicSyndrome'] == 1) & (dataset['Sex'] == 1) & (dataset['WaistCirc'] >= 80)]),
        female_ms_cases
    )
    
    # Blood Glucose Likelihood
    # Ref: American Diabetes Association. Diabetes Care. 2021;44(Suppl 1):S15-S33
    # Fasting glucose ≥ 100 mg/dL indicates prediabetes
    likelihoods['glucose_risk'] = safe_division(
        len(dataset[(dataset['MetabolicSyndrome'] == 1) & (dataset['BloodGlucose'] >= 100)]),
        total_cases
    )
    
    # HDL Likelihoods
    # Ref: National Cholesterol Education Program (NCEP) ATP III, 2002
    likelihoods['hdl_male_risk'] = safe_division(
        len(dataset[(dataset['MetabolicSyndrome'] == 1) & (dataset['Sex'] == 0) & (dataset['HDL'] < 40)]),
        male_ms_cases
    )
    
    likelihoods['hdl_female_risk'] = safe_division(
        len(dataset[(dataset['MetabolicSyndrome'] == 1) & (dataset['Sex'] == 1) & (dataset['HDL'] < 50)]),
        female_ms_cases
    )
    
    # Triglycerides Likelihood
    # Ref: NCEP ATP III Guidelines, 2002
    likelihoods['triglycerides_risk'] = safe_division(
        len(dataset[(dataset['MetabolicSyndrome'] == 1) & (dataset['Triglycerides'] >= 150)]),
        total_cases
    )
    
    # Albuminuria Likelihood
    # Ref: Karalliedde J, Viberti G. Am J Med. 2005;118(12):1416-1421
    # Microalbuminuria is an early marker of metabolic syndrome
    likelihoods['albuminuria_risk'] = safe_division(
        len(dataset[(dataset['MetabolicSyndrome'] == 1) & (dataset['Albuminuria'] == 1)]),
        total_cases
    )
    
    # UrAlbCr Likelihood
    # Ref: KDIGO 2012 Clinical Practice Guideline
    # Urine albumin-to-creatinine ratio ≥ 30 mg/g indicates kidney damage
    likelihoods['uralbcr_risk'] = safe_division(
        len(dataset[(dataset['MetabolicSyndrome'] == 1) & (dataset['UrAlbCr'] >= 30)]),
        total_cases
    )
    
    # Uric Acid Likelihoods
    # Ref: Borghi C, et al. J Hypertens. 2015;33(9):1729-1741
    likelihoods['uric_acid_male_risk'] = safe_division(
        len(dataset[(dataset['MetabolicSyndrome'] == 1) & (dataset['Sex'] == 0) & (dataset['UricAcid'] > 7.0)]),
        male_ms_cases
    )
    
    likelihoods['uric_acid_female_risk'] = safe_division(
        len(dataset[(dataset['MetabolicSyndrome'] == 1) & (dataset['Sex'] == 1) & (dataset['UricAcid'] > 6.0)]),
        female_ms_cases
    )
    
    return likelihoods

def calculate_evidence_based_posteriors(dataset, likelihoods):
    """
    Calculate posterior probabilities using Bayes' Theorem
    """
    prior = len(dataset[dataset['MetabolicSyndrome'] == 1]) / len(dataset)
    total_samples = len(dataset)
    
    posteriors = {}
    
    def calculate_feature_prob(condition):
        return len(dataset[condition]) / total_samples
    
    feature_conditions = {
        'age_male_risk': (dataset['Sex'] == 0) & (dataset['Age'] >= 40),
        'age_female_risk': (dataset['Sex'] == 1) & (dataset['Age'] >= 51),
        'bmi_risk': dataset['BMI'] >= 30,
        'waist_male_risk': (dataset['Sex'] == 0) & (dataset['WaistCirc'] >= 94),
        'waist_female_risk': (dataset['Sex'] == 1) & (dataset['WaistCirc'] >= 80),
        'glucose_risk': dataset['BloodGlucose'] >= 100,
        'hdl_male_risk': (dataset['Sex'] == 0) & (dataset['HDL'] < 40),
        'hdl_female_risk': (dataset['Sex'] == 1) & (dataset['HDL'] < 50),
        'triglycerides_risk': dataset['Triglycerides'] >= 150,
        'albuminuria_risk': dataset['Albuminuria'] == 1,
        'uralbcr_risk': dataset['UrAlbCr'] >= 30,
        'uric_acid_male_risk': (dataset['Sex'] == 0) & (dataset['UricAcid'] > 7.0),
        'uric_acid_female_risk': (dataset['Sex'] == 1) & (dataset['UricAcid'] > 6.0)
    }
    
    race_mapping = {0: 'White', 1: 'Asian', 2: 'Black', 3: 'MexAmerican', 4: 'Hispanic', 5: 'Other'}
    for race_code, race_name in race_mapping.items():
        feature_conditions[f'race_{race_name.lower()}'] = dataset['Race'] == race_code
    
    for feature, condition in feature_conditions.items():
        p_feature = calculate_feature_prob(condition)
        likelihood = likelihoods.get(feature, 0)
        
        if p_feature > 0:
            posterior = (likelihood * prior) / p_feature
            posteriors[feature] = min(1.0, posterior)
        else:
            posteriors[feature] = 0
    
    return prior, posteriors

def analyze_metabolic_syndrome_evidence_based(dataset):
    """
    Perform evidence-based analysis of metabolic syndrome risk factors
    """
    # Calculate likelihoods
    likelihoods = calculate_evidence_based_likelihoods(dataset)
    
    # Format and print likelihoods
    likelihood_df = pd.DataFrame(likelihoods.items(), columns=['Measure', 'Likelihood'])
    likelihood_df['Likelihood'] = likelihood_df['Likelihood'].round(3)
    likelihood_df['Percentage'] = (likelihood_df['Likelihood'] * 100).round(1).astype(str) + '%'
    print("Evidence-Based Likelihood Analysis:")
    print(likelihood_df.sort_values('Likelihood', ascending=False))
    print("\n")
    
    # Calculate posteriors
    prior, posteriors = calculate_evidence_based_posteriors(dataset, likelihoods)
    
    # Format and print posteriors
    posterior_df = pd.DataFrame(posteriors.items(), columns=['Parameter', 'Posterior'])
    posterior_df['Posterior'] = posterior_df['Posterior'].round(3)
    posterior_df['Percentage'] = (posterior_df['Posterior'] * 100).round(1).astype(str) + '%'
    posterior_df = posterior_df.sort_values('Posterior', ascending=False)
    
    print(f"Prior probability of Metabolic Syndrome: {prior:.3f} ({prior*100:.1f}%)\n")
    print("Evidence-Based Posterior Probabilities by Parameter:")
    print(posterior_df)
    
    return likelihood_df, posterior_df

In [ ]:
import pandas as pd

# Load your dataset
dataset = pd.read_csv('dataset.csv')

# Run the analysis
likelihood_df, posterior_df = analyze_metabolic_syndrome_evidence_based(dataset)
print("Likelihood: ")
print(likelihood_df)
print("\n\n")
print("Posterior:")
print(posterior_df)